[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/research_assistant/openai_agents.ipynb)

# Research assistant — OpenAI Agents SDK

Typed Agents and Runner stages use a thin deterministic SDK Model adapter. Specialist handoffs are declared; validated Python controls retrieval and transfer of evidence. No API key is needed. Mock runtime is under one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai-agents==0.17.8", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar

from agents import Agent, Runner, set_tracing_disabled
from agents.items import ModelResponse as SDKResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import ResponseOutputMessage, ResponseOutputText
from pydantic import BaseModel, ConfigDict, Field

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.safety import SafetyEngine, SafetyPolicy, UntrustedContent  # noqa: E402
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture = json.loads((ROOT / "data/research_assistant/case_mock.json").read_text())

## Typed agents, handoffs and adapter
The adapter translates response shape only. Planner, extractor, writer and critic instructions remain visible. Retrieved instructions are isolated before they can reach another agent.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: str


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: str


Ledger.model_rebuild(_types_namespace={"Claim": Claim})


def core_model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


class FixtureModel(SDKModel):
    def __init__(self):
        self.core = core_model()

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        response = await self.core.generate([user(str(input))])
        text = json.dumps(response.structured_output, sort_keys=True)
        item = ResponseOutputMessage(
            id=response.response_id,
            content=[
                ResponseOutputText(annotations=[], text=text, type="output_text", logprobs=[])
            ],
            role="assistant",
            status="completed",
            type="message",
        )
        return SDKResponse(output=[item], usage=Usage(), response_id=response.response_id)

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


async def run_agent(name, instructions, model, output_type, prompt, handoffs=None):
    result = await Runner.run(
        Agent(
            name=name,
            instructions=instructions,
            model=model,
            output_type=output_type,
            handoffs=handoffs or [],
        ),
        prompt,
        max_turns=2,
    )
    return result.final_output


def search(query):
    terms = set(query.casefold().split())
    return [
        r
        for r in catalogue["records"]
        if terms & set((r["title"] + " " + r["passage"]).casefold().split())
    ]

In [ ]:
async def run_research():
    model = FixtureModel()
    trace = []
    extractor = Agent(
        name="Evidence extractor",
        instructions="Extract only source-grounded claims.",
        model=model,
        output_type=Ledger,
    )
    critic_agent = Agent(
        name="Evidence critic",
        instructions="Reject unsupported or uncited synthesis.",
        model=model,
        output_type=CriticDecision,
    )
    plan = await run_agent(
        "Planner", "Plan at most four bounded searches.", model, QueryPlan, "Plan.", [extractor]
    )
    trace.append({"event": "plan"})
    retrieved = {r["source_id"]: r for q in plan.queries for r in search(q)}
    safety = SafetyEngine(SafetyPolicy())
    assessments = {
        sid: safety.inspect_retrieved_content(UntrustedContent(source_id=sid, text=r["passage"]))
        for sid, r in retrieved.items()
    }
    safe = {sid: r for sid, r in retrieved.items() if not assessments[sid].indicators}
    trace.append(
        {"event": "retrieve", "isolated": [sid for sid, a in assessments.items() if a.indicators]}
    )
    ledger_result = await Runner.run(extractor, f"Extract only from {safe}", max_turns=2)
    claims = tuple(c for c in ledger_result.final_output.claims if c.source_id in safe)
    trace.append(
        {"event": "ledger", "conflicts": [c.source_id for c in claims if c.stance == "conflicting"]}
    )
    answer = await run_agent(
        "Writer",
        "Report conditions and conflicts with source IDs.",
        model,
        Synthesis,
        str(claims),
        [critic_agent],
    )
    trace.append({"event": "synthesis"})
    critic_result = await Runner.run(critic_agent, str(answer.model_dump()), max_turns=2)
    valid = set(answer.source_ids).issubset(safe)
    trace.append(
        {
            "event": "critic",
            "accepted": critic_result.final_output.accepted,
            "citations_valid": valid,
        }
    )
    return {
        "claims": claims,
        "answer": answer,
        "outcome": answer.outcome
        if critic_result.final_output.accepted and valid
        else "abstention",
        "trace": trace,
        "model_calls": 4,
        "termination": "criteria_met",
    }


first = await run_research()
second = await run_research()
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 5 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["answer"].source_ids,
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 4},
    "fallback": "invalid handoff evidence produces abstention",
}

## Limitation
The fixture adapter verifies SDK orchestration, not hosted-model research quality; SDK response types are version-sensitive.